# M33 — Build Semantic Search

**Objective:** build an end-to-end embedding search service.

M28 ranked bundled sentence vectors. The useful whole here is a
**retriever**:

`query → embed (load, do not download) → exact cosine index → ranked
evidence (document/chunk IDs, scores, text spans, provenance)`

Every index carries identity: embedding model/version/metric, corpus
hash, filter schema, and a stale-index policy. High cosine is a
retrieval score. It is not labeled relevance and it is not an answer.

This notebook uses `v06-teaching-meanpool` version `v06.1` and an
**exact in-memory** baseline. Nothing is downloaded. Generation,
reranking, ANN infrastructure, and attention stay closed (M34, M35,
M36, M29).


## Working contract

Every experiment follows **predict → act → observe → explain**. **Predict before running**
each action cell and timestamp the prediction in your own evidence log.
A prediction is falsifiable: a chunk id, a candidate count, a hash
mismatch, or a refused search.

Do not download an encoder, do not treat a cosine as an answer, and do
not open a required vector-database service. If a failure can be
diagnosed from index metadata, stay there.

The repository does not prefill learner answers, ADR text, or competence.


In [ ]:
from pathlib import Path
import inspect
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "missions" / "M33" / "semantic_search.py").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Run from the LearningOS-AI repository or its labs directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from missions.M33.semantic_search import (
    FilterSchemaError,
    IndexIncompatibleError,
    IndexStaleError,
    QueryEmbedding,
    build_index,
    encode_query,
    evaluate_labeled,
    exact_cosine_rank,
    l2_norm,
    load_canonical_corpus,
    load_canonical_index,
    load_encoder,
    load_expected_payload,
    load_incompatible_index,
    load_query_map,
    rebuild_index,
    replace_chunk_text,
    search,
    search_labeled,
    search_report,
    search_unchecked,
    source_hash,
)

corpus = load_canonical_corpus()
index = load_canonical_index()
encoder = load_encoder()
QUERIES = load_query_map()
EXPECTED = load_expected_payload()
incompatible = load_incompatible_index()

print("repository root:", ROOT)
print("index_id:", index.metadata.index_id)
print("backend:", index.metadata.backend)
print("model:", index.metadata.embedding.model)
print("version:", index.metadata.embedding.version)
print("metric:", index.metadata.embedding.metric)
print("normalization:", index.metadata.embedding.normalization)
print("documents:", index.metadata.document_count, "chunks:", index.metadata.chunk_count)
print("downloaded:", index.metadata.downloaded, "network:", index.metadata.network_required)
print("source_hash:", index.metadata.source_hash[:16])


## M28 boundary: vectors in, search service here

M28 already attached offline sentence vectors and refused mixed
provenance. M33 asks what happens when those vectors become an
**index** with document identity, filters, labeled evaluation, and a
staleness gate.

What this mission **opens:** document/chunk IDs, text spans, embedding
load, exact similarity, top-k, metadata filters, query path, latency,
labeled retrieval, index version/hash.

What stays **deferred:**

- M29 — attention; queries, keys, values; softmax
- M34 — generation, citations, abstention
- M35 — reranking and chunk-size tuning
- M36 — ANN, required Qdrant infrastructure, hybrid fusion

M28 cosine ranking is not re-taught. The exact baseline here is the
same metric on a versioned chunk store.

Canonical sources: `sentence-transformers` and `qdrant-docs` in
`data/source_registry.json`. Skip production encoders and hosted
indexes here.


## Frozen teaching fixtures

Declare the useful whole **before** the first search.

| Fixture | Value |
| --- | --- |
| Index | `v08-exact-memory` |
| Backend | exact in-memory |
| Corpus | `m33.corpus.v1` (7 documents, 14 chunks) |
| Embedding | `v06-teaching-meanpool` `v06.1` |
| Width | 12 |
| Pooling / norm / metric | mean / L2 / cosine |
| Filters | `topic`, `source`, `locale` |
| Ties | `(-score, chunk_id)` |
| Stale policy | fail closed; rebuild or reject |
| Canonical query | `I forgot my password and cannot sign in.` |
| Download | false — JSON in `datasets/M33/` copied from M28 |

The wrapping is new: M28 item `d-password-forgot` is now chunk
`doc-account-access::c0` with a recovered character span. Labels live
in `queries.json` and are **not** cosine ranks.


In [ ]:
print("fingerprint", index.metadata.fingerprint())
print("--- documents ---")
for document in corpus.documents:
    meta = document.metadata_dict()
    print(f"{document.document_id:22s} topic={meta['topic']:8s} chunks={len(document.chunks)}")
    for chunk in document.chunks:
        recovered = document.text[chunk.span_start:chunk.span_end]
        print(f"  {chunk.chunk_id:24s} [{chunk.span_start:3d}:{chunk.span_end:3d}] {chunk.text}")
        assert recovered == chunk.text
print("--- labeled queries ---")
for query in QUERIES.values():
    print(f"{query.query_id:16s} relevant={list(query.relevant_chunk_ids)} :: {query.text}")
assert corpus.embedding.downloaded is False
assert index.metadata.embedding.version == "v06.1"
assert incompatible.metadata.embedding.version == "v06.2"
assert index.metadata.chunk_count == 14


### Identity is part of the contract

A hit is only valid against an index with the same embedding
fingerprint **and** the same corpus `source_hash`. If either changes,
scores are not comparable to the live source. That is the migration
trigger M34 will inherit: evidence must name the index, not just a
cosine.

`topic=account` on a chunk is eligibility metadata. It is not a
guarantee that a production encoder has an "account neuron."


In [ ]:
sample = index.get("doc-account-access::c0")
query_vec = encode_query(QUERIES["q-password"].text, query_id="q-password")
print("chunk", sample.chunk.chunk_id, "doc", sample.chunk.document_id)
print("span", sample.chunk.span())
print("vector shape", (len(sample.vector),), "norm", round(l2_norm(sample.vector), 6))
print("query shape", (len(query_vec.vector),), "norm", round(l2_norm(query_vec.vector), 6))
print("query model", query_vec.provenance.model, query_vec.provenance.version)
print("index hash", index.metadata.source_hash)
print("live hash ", source_hash(corpus))
assert abs(l2_norm(sample.vector) - 1.0) < 1e-9
assert query_vec.provenance.version == index.metadata.embedding.version
assert source_hash(corpus) == index.metadata.source_hash


### A stored chunk is a contract row

Width 12, L2 norm 1, model `v06-teaching-meanpool`, version `v06.1`,
and a source hash that currently matches the live corpus. Those facts
are data. The next cells search these rows without downloading anything.


## Predict before running — first ranked evidence

Timestamp a prediction before `run-search`.

Query `q-password`: `I forgot my password and cannot sign in.`

The index is frozen. Predict:

- the **top document_id** (not just "an account row")
- whether a printer chunk can appear in the top three
- which evidence fields a later grounding stage would need besides
  the score

Do not compute cosine yet. A guess from the texts is the point.


In [ ]:
password_hits = search_labeled("q-password", top_k=5, enforce_freshness=True)
print(search_report(password_hits))
for hit in password_hits.hits:
    print(hit.rank, hit.chunk_id, hit.document_id, round(hit.score, 4), hit.span_start, hit.span_end, hit.text)
print("evidence row", password_hits.hits[0].as_evidence())
assert password_hits.ids()[:3] == tuple(EXPECTED["q-password_top3"])
assert "doc-device-printer::c0" not in password_hits.ids()[:3]
assert password_hits.hits[0].as_evidence()["document_id"] == "doc-account-access"
print("scored_candidates", password_hits.scored_candidates)


### Ranked evidence is not an answer

The password query sits on account chunks. Printer rows exist in the
same index and lose. Cosine did not "understand a login problem." It
ranked stored vectors under a declared metric and returned spans that
M34 could pack later. This notebook does not generate a reply.


## Predict before running — top-k window

Timestamp a prediction before `run-topk`.

**Change:** vary `top_k` on the same `q-password` vectors (try 1, 3, and 8).

**Invariant:** index and query vectors stay fixed.

Predict:

- whether `scored_candidates` changes with k
- whether recall opportunity can rise while extra irrelevant ids enter
  the window


In [ ]:
encoded_password = encode_query(QUERIES["q-password"].text, query_id="q-password")
topk_rows = []
for k in (1, 3, 8):
    result = search(index, encoded_password, top_k=k, live_corpus=corpus, enforce_freshness=True)
    metrics = evaluate_labeled(result, QUERIES["q-password"], k=k)
    topk_rows.append((k, result, metrics))
    print(
        "k", k,
        "returned", result.ids(),
        "scored", result.scored_candidates,
        "recall", metrics["recall_at_k"],
        "hit", metrics["hit_at_k"],
    )
assert topk_rows[0][1].scored_candidates == topk_rows[-1][1].scored_candidates == 14
assert topk_rows[0][2]["recall_at_k"] <= topk_rows[-1][2]["recall_at_k"]
assert len(topk_rows[0][1].hits) == 1
assert len(topk_rows[-1][1].hits) == 8
password_k3 = topk_rows[1][1]


In [ ]:
ks = list(range(1, 9))
recalls = []
irrelevant = []
for k in ks:
    result = search(index, encoded_password, top_k=k, live_corpus=corpus, enforce_freshness=True)
    metrics = evaluate_labeled(result, QUERIES["q-password"], k=k)
    recalls.append(metrics["recall_at_k"])
    relevant = set(QUERIES["q-password"].relevant_chunk_ids)
    irrelevant.append(sum(1 for item_id in result.ids() if item_id not in relevant))
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(ks, recalls, marker="o", label="recall@k (labels)")
ax.plot(ks, [n / 3 for n in irrelevant], marker="s", label="irrelevant in window / 3")
ax.set_xlabel("top-k")
ax.set_ylabel("rate")
ax.set_title("Password query: does a larger k add labels or noise?")
ax.set_ylim(-0.05, 1.05)
ax.set_xticks(ks)
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()
print("recall by k", list(zip(ks, recalls)))
print("irrelevant counts", list(zip(ks, irrelevant)))


### Exact search scores everyone; top-k only cuts the list

On this tiny corpus, k does not change how many rows are scored. It
changes how many of those scores you look at. That is why recall can
rise while the window also admits chunks the labels did not mark
relevant. Approximate indexes (later) change *who gets scored*. That
is not this mission.


## Predict before running — metadata filter

Timestamp a prediction before `run-filter`.

**Change:** run the same `q-password` query with `topic=account`, then
with `topic=device`, then with no filter.

**Invariant:** same query vector and same index.

Predict:

- which document ids are allowed to compete under each filter
- whether labeled relevance changes when the filter changes
- what happens if you pass a filter key that is not in the schema


In [ ]:
filter_none = search(index, encoded_password, top_k=5, live_corpus=corpus, enforce_freshness=True)
filter_account = search(
    index, encoded_password, top_k=5, filters={"topic": "account"}, live_corpus=corpus, enforce_freshness=True
)
filter_device = search(
    index, encoded_password, top_k=5, filters={"topic": "device"}, live_corpus=corpus, enforce_freshness=True
)
print("none   ", filter_none.ids(), "scored", filter_none.scored_candidates)
print("account", filter_account.ids(), "scored", filter_account.scored_candidates)
print("device ", filter_device.ids(), "scored", filter_device.scored_candidates)
print("device eval vs password labels", evaluate_labeled(filter_device, QUERIES["q-password"], k=2))
assert filter_account.scored_candidates == 3
assert filter_device.scored_candidates == 2
assert list(filter_device.ids()[:2]) == EXPECTED["filter_topic_device_top2"]
assert all(hit.metadata_dict()["topic"] == "device" for hit in filter_device.hits)
assert QUERIES["q-password"].relevant_chunk_ids == (
    "doc-account-access::c0",
    "doc-account-access::c1",
    "doc-account-access::c2",
)
try:
    search(index, encoded_password, filters={"channel": "email"}, live_corpus=corpus, enforce_freshness=True)
    raise AssertionError("expected FilterSchemaError")
except FilterSchemaError as exc:
    print("schema rejection", exc)


### Filters change eligibility, not truth

A device-only filter can make printer chunks win a password query
because they are the only rows allowed to compete. The labels did not
change. Do not "fix" that by relabeling printers as relevant. Payload
filters in a later vector store should preserve this eligibility rule;
this mission implements it as a pre-score mask on the exact baseline.


## Predict before running — exact baseline parity

Timestamp a prediction before `run-baseline`.

**Change:** compute an independent brute-force cosine ranking on the
same unit vectors and compare it to the service.

**Invariant:** same vectors, same metric, same filter mask.

Predict whether any id or score can differ, including ties.


In [ ]:
def independent_rank(query_vector, records, top_k, filters=None):
    query = np.asarray(query_vector, dtype=float)
    query = query / np.linalg.norm(query)
    rows = []
    for record in records:
        meta = dict(record.chunk.metadata)
        if filters and any(meta.get(key) != value for key, value in filters.items()):
            continue
        vector = np.asarray(record.vector, dtype=float)
        vector = vector / np.linalg.norm(vector)
        score = float(np.clip(float(np.dot(query, vector)), -1.0, 1.0))
        rows.append((record.chunk.chunk_id, score))
    rows.sort(key=lambda item: (-item[1], item[0]))
    return rows[:top_k]

service_rank = exact_cosine_rank(encoded_password.vector, index.records, top_k=8)
oracle = independent_rank(encoded_password.vector, index.records, 8)
print("service", [row[0].chunk.chunk_id for row in service_rank])
print("oracle ", [row[0] for row in oracle])
assert [row[0].chunk.chunk_id for row in service_rank] == [row[0] for row in oracle]
for left, right in zip(service_rank, oracle, strict=True):
    assert abs(left[1] - right[1]) < 1e-12
device_service = search(index, encoded_password, top_k=5, filters={"topic": "device"}, live_corpus=corpus)
device_oracle = independent_rank(encoded_password.vector, index.records, 5, filters={"topic": "device"})
assert list(device_service.ids()) == [row[0] for row in device_oracle]
print("parity ok; filtered oracle", [row[0] for row in device_oracle])


### Exact means "this arithmetic," not "a vendor API"

Parity is the contract M36 will need as a correctness reference. If a
later approximate index disagrees, the disagreement is a recall/latency
trade-off against this ranking, not a new definition of cosine.


## Predict before running — labeled retrieval

Timestamp a prediction before `run-labels`.

**Change:** compute hit@k and recall@k from `queries.json`, not from
the cosine column.

**Invariant:** evaluation set stays frozen.

Predict whether a top-scoring chunk is always labeled relevant, and
what you would do if it is not (hint: not relabeling).


In [ ]:
label_rows = []
for query_id in ("q-password", "q-paraphrase", "q-printer", "q-low-overlap"):
    result = search_labeled(query_id, top_k=3, enforce_freshness=True)
    metrics = evaluate_labeled(result, QUERIES[query_id], k=3)
    label_rows.append(metrics)
    print(query_id, "ids", result.ids()[:3], "hit", metrics["hit_at_k"], "recall", metrics["recall_at_k"], "top_relevant", metrics["top_id_labeled_relevant"])
assert label_rows[0]["recall_at_k"] == 1.0
assert label_rows[0]["scores_are_not_labels"] is True
print("labels remain", {qid: list(QUERIES[qid].relevant_chunk_ids) for qid in ("q-password", "q-printer")})


### Score is a ranking signal; labels are a judgment

hit@k asks whether any labeled chunk appeared. recall@k asks how many
of the labeled chunks appeared. Neither number is the cosine itself.
A later generator can still be wrong when retrieval is right, and
right-looking prose can be wrong when retrieval missed. That split is
M34's job. Keep the labels still.


## Predict before running — inherited hard cases

Timestamp a prediction before `run-hard`.

**Change:** run the M28 negation, numeric, and entity queries through
the service. Domain query stays in-domain.

**Invariant:** evaluation set and encoder stay fixed.

Predict whether the unlabeled neighbor can still score high, and
whether you will change any relevant ids after seeing the ranks.


In [ ]:
negation_hits = search_labeled("q-negation", top_k=3, enforce_freshness=True)
numeric_hits = search_labeled("q-numeric", top_k=3, enforce_freshness=True)
entity_hits = search_labeled("q-entity", top_k=3, enforce_freshness=True)
domain_hits = search_labeled("q-domain", top_k=3, enforce_freshness=True)
print("negation", search_report(negation_hits)["hits"][:2])
print("numeric ", [ (h.chunk_id, round(h.score, 4)) for h in numeric_hits.hits[:2] ])
print("entity  ", [ (h.chunk_id, round(h.score, 4)) for h in entity_hits.hits[:2] ])
print("domain  ", domain_hits.ids()[:3])
neg_eval = evaluate_labeled(negation_hits, QUERIES["q-negation"], k=2)
print("negation eval k=2", {k: neg_eval[k] for k in ("hit_at_k", "recall_at_k", "precision_at_k", "top_id_labeled_relevant")})
assert list(negation_hits.ids()[:2]) == EXPECTED["q-negation_top2"]
assert negation_hits.hits[1].score > 0.85
assert negation_hits.hits[1].chunk_id not in QUERIES["q-negation"].relevant_chunk_ids
assert list(numeric_hits.ids()[:2]) == EXPECTED["q-numeric_top2"]
assert list(entity_hits.ids()[:2]) == EXPECTED["q-entity_top2"]
assert domain_hits.top_id == "doc-weather::c0"
assert "doc-legal::c0" not in domain_hits.ids()[:2]
print("labels after inspection", QUERIES["q-negation"].relevant_chunk_ids)


### High cosine is cheap to over-read

Deny versus approve share refund mass; fifty versus thousand share
payment mass; ticket 4412 versus 4413 almost overlap. The service
will still return those neighbors with high scores. The labels still
name one relevant chunk. Do not promote the neighbor into the relevant
set because the number looked confident.


## Predict before running — latency and candidate count

Timestamp a prediction before `run-latency`.

**Change:** measure `latency_ms` and `scored_candidates` for k=1 versus
k=8, then for an unfiltered search versus `topic=legal`.

**Invariant:** same index. Exact search still walks eligible rows.

Predict which knob actually reduces work on this baseline.


In [ ]:
lat_k1 = search(index, encoded_password, top_k=1, live_corpus=corpus, enforce_freshness=True)
lat_k8 = search(index, encoded_password, top_k=8, live_corpus=corpus, enforce_freshness=True)
lat_legal = search(
    index, encoded_password, top_k=8, filters={"topic": "legal"}, live_corpus=corpus, enforce_freshness=True
)
print("k1    scored", lat_k1.scored_candidates, "ms", round(lat_k1.latency_ms, 3), "ids", lat_k1.ids())
print("k8    scored", lat_k8.scored_candidates, "ms", round(lat_k8.latency_ms, 3), "n", len(lat_k8.hits))
print("legal scored", lat_legal.scored_candidates, "ms", round(lat_legal.latency_ms, 3), "ids", lat_legal.ids())
assert lat_k1.scored_candidates == lat_k8.scored_candidates == 14
assert lat_legal.scored_candidates == 1
assert lat_k1.latency_ms >= 0.0
print("milliseconds are a teaching trace, not a production SLA")


### Work is eligibility, not the printed k

k=1 and k=8 score the same 14 rows. A metadata filter scores fewer
rows. Treat the printed milliseconds as instrumentation on a toy
corpus, not as a reason to skip the exact baseline.


## Code reading — ingest, index, query, filter, score, ties, freshness

Read `compose_document`, `build_index`, `search`, `exact_cosine_rank`,
`assert_fresh`, and `rebuild_index` in `missions/M33/semantic_search.py`.

**Predict before running** the next cell:

1. whether changing `top_k` from 1 to 8 changes `scored_candidates`
   on an unfiltered exact search
2. whether `document.text[span]` equals the chunk text
3. whether `search` on a mutated live corpus raises before it returns
   hits when `enforce_freshness=True`

Do not search the file for a generator, a reranker, or an approximate
index class. Stay on identity, cosine, filters, and the hash gate.


In [ ]:
search_src = inspect.getsource(search)
build_src = inspect.getsource(build_index)
rank_src = inspect.getsource(exact_cosine_rank)
print("search mentions enforce_freshness", "enforce_freshness" in search_src)
print("search mentions enforce_provenance", "enforce_provenance" in search_src)
print("rank sorts by (-score, chunk_id)", "(-item[1], item[0].chunk.chunk_id)" in rank_src)
print("build stores source_hash", "source_hash" in build_src)
account = corpus.get_document("doc-account-access")
chunk0 = account.chunks[0]
print("span recovers text", account.text[chunk0.span_start:chunk0.span_end] == chunk0.text)
print("scored_candidates vs k", lat_k1.scored_candidates, lat_k8.scored_candidates)
print("model", encoder.provenance.model, "version", encoder.provenance.version, "downloaded", encoder.provenance.downloaded)


## Predict before running — Controlled failure: stale index

Timestamp a prediction before `run-failure`.

Build is already done. **Change:** replace live text for
`doc-account-access::c0` with `Please reset the printer firmware.`
Do **not** rebuild.

Query `q-password` stays fixed. The defective path is one named
change: `enforce_freshness=False` (`search_unchecked`).

Predict:

- whether unchecked search still returns a plausible ranking
- whether the served chunk text matches the **live** text or the
  **indexed** text
- whether the live `source_hash` still matches the index


In [ ]:
live_stale = replace_chunk_text(
    corpus,
    "doc-account-access::c0",
    "Please reset the printer firmware.",
)
print("index hash", index.metadata.source_hash[:16])
print("live hash ", source_hash(live_stale)[:16])
print("live text ", live_stale.get_chunk("doc-account-access::c0").text)
defective = search_unchecked(
    index,
    encoded_password,
    top_k=3,
    live_corpus=live_stale,
)
print(search_report(defective))
for hit in defective.hits[:3]:
    print(hit.rank, hit.chunk_id, round(hit.score, 4), hit.text)
assert defective.enforced_freshness is False
assert defective.top_id == "doc-account-access::c0"
assert defective.hits[0].text == "I forgot my password and cannot sign in."
assert defective.hits[0].text != live_stale.get_chunk("doc-account-access::c0").text
print("canonical password top", password_hits.top_id)


### Diagnose before repair

Symptom: a ranking came back with scores that look like similarities,
and the top chunk still reads as a password reset even though the live
corpus now talks about printer firmware.

Hypotheses worth separating: the query text changed; cosine is
undefined; the served row is a stale copy; the embedding model
changed.

The discriminating observation is the hash pair plus the served text.
The index `source_hash` does not match the live corpus, and the hit
text equals the **indexed** string. The root cause is serving an index
that was not rebuilt, not a mysterious encoder.

Do not repair this by editing the query, relabeling, or opening M34–M36.


## Predict before running — smallest repair

Timestamp a prediction before `run-failure-repair`.

Predict that `search(..., enforce_freshness=True)` on the **same**
stale index object and live corpus:

- raises `IndexStaleError`
- and that `rebuild_index(index, live_stale)` then searches using the
  new text/vectors

Do not repair by swapping printer axes inside the scorer.


In [ ]:
try:
    search(index, encoded_password, top_k=3, live_corpus=live_stale, enforce_freshness=True)
    raise AssertionError("expected IndexStaleError")
except IndexStaleError as exc:
    print("raised", exc)
    print("index hash", exc.index_source_hash[:16], "live", exc.live_source_hash[:16])
    repaired_error = exc

rebuilt = rebuild_index(index, live_stale)
print("rebuilt hash", rebuilt.metadata.source_hash[:16])
repaired_hits = search(rebuilt, encoded_password, top_k=3, live_corpus=live_stale, enforce_freshness=True)
print(search_report(repaired_hits))
for hit in repaired_hits.hits[:3]:
    print(hit.rank, hit.chunk_id, round(hit.score, 4), hit.text)
assert repaired_hits.enforced_freshness is True
assert repaired_hits.top_id != "doc-account-access::c0"
still_broken = search_unchecked(index, encoded_password, top_k=1, live_corpus=live_stale)
assert still_broken.hits[0].text == "I forgot my password and cannot sign in."

mixed = search_unchecked(
    incompatible,
    encoded_password,
    top_k=3,
    enforce_provenance=False,
    live_corpus=corpus,
)
print("unchecked mixed ids", mixed.ids())
try:
    search(incompatible, encoded_password, top_k=3, live_corpus=corpus, enforce_provenance=True)
    raise AssertionError("expected IndexIncompatibleError")
except IndexIncompatibleError as exc:
    print("incompatible mismatches", exc.mismatches)
    incompatible_error = exc
assert "version" in incompatible_error.mismatches
print("freshness and provenance gates restored")


### Rebuild or reject; do not reinterpret stale hits

The smallest repair uses the broken objects: the old index still
serves old text when unchecked; the gate refuses it; rebuild re-ingests
the live corpus. Logging model, version, metric, and `source_hash` is
how M34 will know which evidence it packed. Serving stale rows because
the cosine "looked fine" is the other ADR option — not this repair.


## Evidence contract

Submit, in your own log (not in this repository):

- timestamped **Predict before running** notes
- first ranked-evidence trace with ids, spans, and provenance
- top-k sweep with scored_candidates
- filter on/off eligibility
- independent brute-force parity
- labeled hit/recall without relabeling hard cases
- stale-index diagnosis and rebuild-or-reject repair

See `missions/M33/evidence_contract.yaml`. Do not paste filled evidence
into the committed notebook.


## No-AI gate

Close this notebook and complete `missions/M33/no_ai_gate.md` from a blank
page without AI-generated code, calculations, prose, or diagrams.

Use only `datasets/M33/transfer.json`. Rank by cosine, trace one query,
compute labeled success, diagnose the stale probe, and state why a high
similarity score is not answer correctness.

**Status:** [UNFILLED BY LEARNER]


## Unfilled ADR

Use `missions/M33/adr_prompt.md` to choose a V08 index identity and
rebuild policy (embedding version, corpus hash, metric, filter schema,
ties, stale handling). Do not claim a production vector database.

- **Status:** [UNFILLED BY LEARNER]
- **Date:** [UNFILLED BY LEARNER]
- **Owner:** [UNFILLED BY LEARNER]
- **Decision:** [UNFILLED BY LEARNER]

This notebook is not that ADR. Formal engineering review is required
at M33 for the package, not as a substitute for the learner ADR.


## M28 → M33 handoff

M28 supplied the embedding contract and hard-case fixtures. M33 indexes
those vectors as a deterministic exact retriever.

M34 may pack `RankedHit.as_evidence()` rows into a context window. It
must not treat a cosine as an answer, and it must not skip this
retriever boundary.

M35/M36 may measure chunking, rerank, or approximate search only after
this exact baseline and these labels are defended.

Reusable artifacts: `search`, `as_evidence`, `corpus.json`,
`queries.json`, `incompatible_vectors.json`, `transfer.json`.


## Mission summary prompt

In your own words, using only observations from this lab:

1. Why is exact search's `scored_candidates` independent of top-k here?
2. How did a topic filter change who competed without changing labels?
3. Why can approve-refund still score above 0.85 under a deny query?
4. What must M34 receive that a ranking without IDs, spans, and
   provenance cannot provide?

Leave the answers in your evidence log, not in this file.


In [ ]:
assert password_hits.ids()[:3] == tuple(EXPECTED["q-password_top3"])
assert password_k3.ids()[:3] == tuple(EXPECTED["q-password_top3"])
assert topk_rows[0][1].scored_candidates == 14
assert list(filter_device.ids()[:2]) == EXPECTED["filter_topic_device_top2"]
assert list(negation_hits.ids()[:2]) == EXPECTED["q-negation_top2"]
assert negation_hits.hits[1].score > 0.85
assert domain_hits.top_id == "doc-weather::c0"
assert defective.enforced_freshness is False
assert defective.hits[0].text == "I forgot my password and cannot sign in."
assert isinstance(repaired_error, IndexStaleError)
assert repaired_hits.top_id != "doc-account-access::c0"
assert "version" in incompatible_error.mismatches
assert still_broken.hits[0].text == "I forgot my password and cannot sign in."
print("M33 integrity checks passed")
